# XGBoost — Validación reproducible para FRP (versión 4)

**Fuente:** `FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx`, hoja `Dataset_v1_FROZEN` (53 especímenes, 8 campañas). No modifica el Excel. Ejecuta XGBoost en **regresión Vf_kN** y **clasificación FM**, con X1 y X2.

- **LOSO externo** (8 campañas retenidas individualmente) y **GroupKFold interno** (solo campañas de entrenamiento): validación sin fuga entre campañas.
- **Aleatorio externo** 5 folds × 5 repeticiones, con CV interno independiente: evaluación convencional para comparación, **no** prueba de generalización entre campañas.
- Búsqueda de hiperparámetros dentro de cada partición de entrenamiento; exportación de predicciones, parámetros, métricas y versiones.

**Importante:** esta es una implementación reproducible y documentada, no una afirmación de reproducción idéntica del protocolo estadístico de v4. Verifique las semillas y rejillas frente al análisis original antes de sustituir tablas. El conjunto de datos se carga mediante el cuadro de subida de Colab.

In [ ]:
%pip -q install xgboost scikit-learn pandas openpyxl matplotlib

In [ ]:
import os, sys, json, hashlib, platform, warnings, zipfile
from pathlib import Path
import numpy as np, pandas as pd, sklearn, xgboost
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, RepeatedKFold, RepeatedStratifiedKFold, KFold, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier
from IPython.display import display
SEED=20261001
# Cambie a 1 para prueba rápida; deje 5 para la ejecución completa 5×5.
REPETICIONES_ALEATORIAS=5
# Para comprobar solo LOSO, cambie a False.
EJECUTAR_ALEATORIO=True
# Rejilla explícita, igual en ambos protocolos; puede ampliarse después.
GRID={'model__max_depth':[2,3], 'model__learning_rate':[0.03,0.1], 'model__subsample':[0.8,1.0]}
OUT=Path('/content/XGBoost_FRP_v4_resultados') if Path('/content').exists() else Path('XGBoost_FRP_v4_resultados')
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0], '| sklearn',sklearn.__version__,'| xgboost',xgboost.__version__)

In [ ]:
from google.colab import files
uploaded=files.upload()
paths=[Path(k) for k in uploaded if k.lower().endswith('.xlsx')]
assert len(paths)==1, 'Seleccione exactamente el Excel congelado (.xlsx).'
source=paths[0]
raw=source.read_bytes(); sha256=hashlib.sha256(raw).hexdigest()
df=pd.read_excel(source,sheet_name='Dataset_v1_FROZEN',engine='openpyxl')
X1=['bw_mm','d_mm','df_mm','a_d','fc_MPa','fsy_MPa','rho_s','fvy_MPa','rho_sv','tf_mm','wf_mm','sf_mm','ffu_MPa','Ef_GPa','rho_fv','alpha_f_deg','WC']
X2=['bw_mm','df_mm','a_d','fc_MPa','fsy_MPa','rho_s','fvy_MPa','rho_sv','ffu_MPa','Ef_GPa','rho_fv','alpha_f_deg','WC']
sets={'X1_full':X1,'X2_reduced':X2}
assert df.shape==(53,32), f'Se esperaban 53×32, se recibió {df.shape}'
assert df.paper_ref.nunique()==8 and not df.isna().any().any()
assert not df[['paper_ref','specimen']].duplicated().any()
assert set(df.FM.unique())=={'D','R'}
assert all(set(v).issubset(df.columns) for v in sets.values())
assert not ({'Vf_kN','FM','paper_ref','row_id','V_kN'} & set(X1+X2))
assert df['row_id'].is_unique
print('Integridad OK:',len(df),'filas,',df.paper_ref.nunique(),'estudios; SHA256:',sha256)
print('FM:',df.FM.value_counts().to_dict()); print('Estudios:',df.paper_ref.value_counts().sort_index().to_dict())

In [ ]:
def make_pipe(task, features, ytrain):
    prep=ColumnTransformer([('numeric','passthrough',[f for f in features if f!='WC']),
                            ('wrapping',OneHotEncoder(handle_unknown='ignore'),['WC'])])
    if task=='regression':
        model=XGBRegressor(objective='reg:squarederror',n_estimators=200,random_state=SEED,n_jobs=1,verbosity=0)
    else:
        # Solo usa las etiquetas del conjunto de entrenamiento (sin fuga).
        pos=int(np.sum(ytrain==1)); neg=int(np.sum(ytrain==0))
        model=XGBClassifier(objective='binary:logistic',eval_metric='logloss',n_estimators=200,
                            scale_pos_weight=neg/max(pos,1),random_state=SEED,n_jobs=1,verbosity=0)
    return Pipeline([('prep',prep),('model',model)])

def pooled_metrics(task,a,b):
    if task=='regression':
        return dict(MAE=float(mean_absolute_error(a,b)),RMSE=float(np.sqrt(mean_squared_error(a,b))),R2=float(r2_score(a,b)))
    return dict(BA=float(balanced_accuracy_score(a,b)),MacroF1=float(f1_score(a,b,average='macro',zero_division=0)),
                MCC=float(matthews_corrcoef(a,b)))

def run_experiment(task, fs, features, protocol):
    X=df[features]; y=(df.FM=='R').astype(int).to_numpy() if task=='classification' else df.Vf_kN.to_numpy(dtype=float)
    groups=df.paper_ref.to_numpy()
    if protocol=='LOSO': splits=LeaveOneGroupOut().split(X,y,groups)
    elif task=='classification': splits=RepeatedStratifiedKFold(n_splits=5,n_repeats=REPETICIONES_ALEATORIAS,random_state=SEED).split(X,y)
    else: splits=RepeatedKFold(n_splits=5,n_repeats=REPETICIONES_ALEATORIAS,random_state=SEED).split(X,y)
    rows=[]; foldrows=[]; paramrows=[]
    for i,(tr,te) in enumerate(splits):
        if protocol=='LOSO':
            inner=GroupKFold(n_splits=min(5,len(np.unique(groups[tr]))))
            cvgroups=groups[tr]
        else:
            # Folds internos independientes, nunca ven el test externo.
            if task=='classification':
                nmin=int(np.bincount(y[tr],minlength=2).min())
                inner=StratifiedKFold(n_splits=min(4,nmin),shuffle=True,random_state=SEED+i)
            else:inner=KFold(n_splits=4,shuffle=True,random_state=SEED+i)
            cvgroups=None
        pipe=make_pipe(task,features,y[tr]); score='balanced_accuracy' if task=='classification' else 'neg_mean_absolute_error'
        search=GridSearchCV(pipe,GRID,cv=inner,scoring=score,n_jobs=1,refit=True,error_score='raise')
        fitargs={'groups':cvgroups} if cvgroups is not None else {}
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore',message='y_pred contains classes not in y_true')
            search.fit(X.iloc[tr],y[tr],**fitargs)
        predicted=search.predict(X.iloc[te]); met=pooled_metrics(task,y[te],predicted)
        repeat=1 if protocol=='LOSO' else i//5+1
        fold=i+1 if protocol=='LOSO' else i%5+1
        foldrows.append(dict(protocol=protocol,task=task,feature_set=fs,repeat=repeat,fold=fold,
                             heldout_study=str(groups[te][0]) if protocol=='LOSO' else '',
                             n_train=len(tr),n_test=len(te),**met))
        paramrows.append(dict(protocol=protocol,task=task,feature_set=fs,repeat=repeat,fold=fold,
                              best_inner_score=float(search.best_score_),best_params=json.dumps(search.best_params_)))
        for j,a,b in zip(te,y[te],predicted):
            rows.append(dict(protocol=protocol,task=task,feature_set=fs,repeat=repeat,fold=fold,
                             row_id=df.iloc[j].row_id,paper_ref=df.iloc[j].paper_ref,
                             actual=float(a),prediction=float(b)))
        print(protocol,task,fs,f'{i+1} folds terminados',flush=True)
    return rows,foldrows,paramrows

In [ ]:
predictions=[]; folds=[]; params=[]
protocols=['LOSO']+(['random_5fold'] if EJECUTAR_ALEATORIO else [])
for task in ['regression','classification']:
    for fs,features in sets.items():
        for protocol in protocols:
            a,b,c=run_experiment(task,fs,features,protocol)
            predictions+=a; folds+=b; params+=c
            # Guardado incremental: si Colab se desconecta, los experimentos terminados quedan en disco.
            pd.DataFrame(predictions).to_csv(OUT/'predicciones.csv',index=False)
            pd.DataFrame(folds).to_csv(OUT/'metricas_por_fold.csv',index=False)
            pd.DataFrame(params).to_csv(OUT/'hiperparametros_por_fold.csv',index=False)
print('Experimentos terminados:',len(folds),'folds externos')

In [ ]:
p=pd.DataFrame(predictions)
summary=[]
for (protocol,task,fs),g in p.groupby(['protocol','task','feature_set']):
    if protocol=='LOSO':
        assert len(g)==53 and g.row_id.nunique()==53, 'LOSO debe predecir una vez cada espécimen'
        m=pooled_metrics(task,g.actual.to_numpy(),g.prediction.to_numpy())
        summary.append(dict(protocol=protocol,task=task,feature_set=fs,n_predictions=len(g),**m))
    else:
        assert len(g)==53*REPETICIONES_ALEATORIAS, 'Cada repetición debe predecir 53 especímenes'
        for rep,z in g.groupby('repeat'):
            assert len(z)==53 and z.row_id.nunique()==53
            m=pooled_metrics(task,z.actual.to_numpy(),z.prediction.to_numpy())
            summary.append(dict(protocol=protocol,task=task,feature_set=fs,repeat=int(rep),n_predictions=len(z),**m))
s=pd.DataFrame(summary);s.to_csv(OUT/'resumen_por_repeticion.csv',index=False)
agg=s.groupby(['protocol','task','feature_set']).agg({k:['mean','std'] for k in ['MAE','RMSE','R2','BA','MacroF1','MCC'] if k in s.columns})
display(s.round(4));display(agg.round(4))
# Inspección de exclusividad externa LOSO: cada estudio solo aparece en su fold de test.
loso=p[p.protocol=='LOSO']
assert all(g.paper_ref.nunique()==1 for _,g in loso.groupby(['task','feature_set','fold']))
assert len(loso)==53*4
meta=dict(seed=SEED,sha256_workbook=sha256,source_name=source.name,sheet='Dataset_v1_FROZEN',
          n_rows=len(df),n_studies=df.paper_ref.nunique(),X1=X1,X2=X2,grid=GRID,
          repeats_random=REPETICIONES_ALEATORIAS,random_validation=EJECUTAR_ALEATORIO,
          python=sys.version,sklearn=sklearn.__version__,xgboost=xgboost.__version__,
          numpy=np.__version__,pandas=pd.__version__,platform=platform.platform(),
          loso_inner='GroupKFold(5) within outer training studies',
          random_inner='KFold/StratifiedKFold(4) within outer training rows')
(OUT/'protocolo_y_entorno.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('Validaciones de integridad y cobertura LOSO: CORRECTAS')

In [ ]:
zip_path=Path('/content/XGBoost_FRP_v4_resultados.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file(): z.write(f,arcname=f.name)
print('ZIP creado:',zip_path)
files.download(str(zip_path))